# DCGAN Concêntrico — Curando o Dataset de Anime com a Própria IA

**Deep Convolutional GAN** (Radford et al., 2015) aplicada ao **Anime Face Dataset**, combinada com a ideia central do projeto: **filtrar o dataset usando a própria rede** para isolar um subconjunto homogêneo ("concêntrico") e mostrar que ele gera melhor que o dataset cru.

**Anotações de aula que guiam o projeto:**

> *"Não adianta espremer na IA, identificar problema do dataset, usar o de anime... não tem número mínimo necessário de imagens, depende do problema... pegue o dataset e me retorne apenas imagens concêntricas, usar a IA para separar as imagens. Falta de variabilidade nos formatos das figuras facilita. Usa instâncias que guardam a mesma relação de proporção, forma e matiz."*

---

## Pipeline

1. **DCGAN-A** treinado no dataset cru (baseline).
2. **Filtragem via IA**: o *discriminador* do DCGAN-A vira extrator de features → embeddings → K-means → escolhemos o **cluster mais denso** como o grupo concêntrico.
3. **DCGAN-B** treinado do zero só nesse cluster.
4. **Comparação**: evolução visual, curvas de loss, e amostras lado a lado dos dois geradores.

```
Ruído z (100-dim) → [Generator] → Imagem Falsa (3×64×64)
                                         ↓
                                  [Discriminator] → Real / Falso
```

> **Referência:** Radford et al. (2015) — *Unsupervised Representation Learning with Deep Convolutional GANs*

## Bibliotecas e Configuração

In [ ]:
%matplotlib inline

import os, glob, random, time, math
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data
from torch.utils.data import Dataset, Subset
import torchvision.transforms as transforms
import torchvision.utils as vutils

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA disp: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

## Semente Aleatória (Reprodutibilidade)

In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print(f"Semente fixada: {SEED}")

## Hiperparâmetros e Dataset

O `glob` recursivo encontra as imagens em **qualquer subpasta** de `/kaggle/input/` — sem caminho fixo, então não há mais `FileNotFoundError` por slug errado.

In [ ]:
# Encontra imagens em qualquer subpasta — resolve problema de caminho
image_files = (
    glob.glob("/kaggle/input/**/*.jpg",  recursive=True) +
    glob.glob("/kaggle/input/**/*.jpeg", recursive=True) +
    glob.glob("/kaggle/input/**/*.png",  recursive=True)
)
image_files = sorted(image_files)

if not image_files:
    raise FileNotFoundError(
        "Nenhuma imagem encontrada em /kaggle/input/\n"
        "Adicione o Anime Face Dataset (splcher/animefacedataset) via '+ Add Input'."
    )

# Usa um subconjunto para acelerar (DCGAN-A no cru). Suba se tiver tempo.
SAMPLE_SIZE = 15000
if len(image_files) > SAMPLE_SIZE:
    idx = np.random.choice(len(image_files), SAMPLE_SIZE, replace=False)
    image_files = [image_files[i] for i in sorted(idx)]

print(f"{len(image_files):,} imagens em uso")
print(f"   exemplo: {image_files[0]}")

# Hiperparâmetros (estilo DCGAN canônico)
IMAGE_SIZE  = 64
NC          = 3      # canais RGB
NZ          = 100    # dimensão do vetor latente z
NGF         = 64     # filtros base do Gerador
NDF         = 64     # filtros base do Discriminador
LR          = 0.0002
BETA1       = 0.5    # padrão GAN
BATCH_SIZE  = 128
WORKERS     = 2

EPOCHS_A    = 20     # DCGAN-A no dataset cru
EPOCHS_B    = 30     # DCGAN-B no subset concêntrico (menor → treina mais)

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo: {DEVICE}")
if DEVICE.type == "cpu":
    print("AVISO: em CPU isso é inviável. Ative GPU no Kaggle (Settings → Accelerator).")

### Dataset e Transformações

`Resize → CenterCrop → ToTensor → Normalize([-1,1])` — o `Tanh` na saída do gerador produz valores em `[-1,1]`, então a normalização casa.

In [ ]:
class FileListDataset(Dataset):
    """Recebe lista de caminhos — funciona com qualquer estrutura de pastas."""
    def __init__(self, files, transform=None):
        self.files = files
        self.transform = transform
    def __len__(self):
        return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, 0   # rótulo dummy — GAN não usa classes

transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),  # -> [-1, 1]
])

dataset_cru = FileListDataset(image_files, transform=transform)
dataloader_cru = torch.utils.data.DataLoader(
    dataset_cru, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=WORKERS, pin_memory=(DEVICE.type=="cuda"), drop_last=True,
)
print(f"Imagens: {len(dataset_cru):,} | Batches/época: {len(dataloader_cru):,}")

# Amostra real para comparações
real_batch, _ = next(iter(dataloader_cru))
plt.figure(figsize=(8, 8)); plt.axis("off")
plt.title("Amostras reais — Anime Face Dataset")
plt.imshow(np.transpose(vutils.make_grid(real_batch[:64], padding=2, normalize=True), (1,2,0)))
plt.show()

## Arquitetura DCGAN

### Inicialização de pesos (Radford et al.)
- Conv / ConvTranspose: pesos ~ N(0, 0.02)
- BatchNorm: pesos ~ N(1.0, 0.02), bias = 0

### Gerador
```
z (N,100,1,1) → 512×4×4 → 256×8×8 → 128×16×16 → 64×32×32 → 3×64×64 (Tanh)
```
### Discriminador
```
img (N,3,64,64) → 64×32×32 → 128×16×16 → 256×8×8 → 512×4×4 → 1 (Sigmoid)
```
O penúltimo mapa de ativação do discriminador (512×4×4) será reaproveitado como **feature** para a filtragem.

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(NZ, NGF*8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(NGF*8), nn.ReLU(True),            # 512 x 4 x 4
            nn.ConvTranspose2d(NGF*8, NGF*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*4), nn.ReLU(True),            # 256 x 8 x 8
            nn.ConvTranspose2d(NGF*4, NGF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF*2), nn.ReLU(True),            # 128 x 16 x 16
            nn.ConvTranspose2d(NGF*2, NGF, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NGF), nn.ReLU(True),              # 64 x 32 x 32
            nn.ConvTranspose2d(NGF, NC, 4, 2, 1, bias=False),
            nn.Tanh()                                        # 3 x 64 x 64
        )
    def forward(self, x):
        return self.main(x)


class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        # features = tudo menos a última conv -> permite extrair embeddings
        self.features = nn.Sequential(
            nn.Conv2d(NC, NDF, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),                 # 64 x 32 x 32
            nn.Conv2d(NDF, NDF*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*2), nn.LeakyReLU(0.2, inplace=True),  # 128 x 16 x 16
            nn.Conv2d(NDF*2, NDF*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*4), nn.LeakyReLU(0.2, inplace=True),  # 256 x 8 x 8
            nn.Conv2d(NDF*4, NDF*8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(NDF*8), nn.LeakyReLU(0.2, inplace=True),  # 512 x 4 x 4
        )
        self.classifier = nn.Sequential(
            nn.Conv2d(NDF*8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )
    def forward(self, x):
        f = self.features(x)
        return self.classifier(f).view(-1)
    def extract(self, x):
        # vetor de features achatado (512*4*4 = 8192) — usado na filtragem
        return self.features(x).view(x.size(0), -1)


def novo_par():
    g = Generator().to(DEVICE); g.apply(weights_init)
    d = Discriminator().to(DEVICE); d.apply(weights_init)
    return g, d

netG_A, netD_A = novo_par()
n_g = sum(p.numel() for p in netG_A.parameters())
n_d = sum(p.numel() for p in netD_A.parameters())
print(f"Parâmetros — Gerador: {n_g:,} | Discriminador: {n_d:,}")

## Função de Treinamento DCGAN

Ciclo padrão por batch:
1. **Discriminador**: maximiza `log(D(x)) + log(1 - D(G(z)))`
2. **Gerador**: maximiza `log(D(G(z)))` (rótulo real para enganar D)

Adam com `beta1=0.5` (mais estável para GANs). Guardamos snapshots do ruído fixo para a evolução visual.

In [ ]:
criterion = nn.BCELoss()
REAL_LABEL, FAKE_LABEL = 1.0, 0.0

def treinar_dcgan(netG, netD, loader, epochs, tag="", fixed_noise=None):
    optD = optim.Adam(netD.parameters(), lr=LR, betas=(BETA1, 0.999))
    optG = optim.Adam(netG.parameters(), lr=LR, betas=(BETA1, 0.999))
    if fixed_noise is None:
        fixed_noise = torch.randn(64, NZ, 1, 1, device=DEVICE)

    G_losses, D_losses, img_list = [], [], []
    print(f"\n=== Treinando {tag} ({epochs} épocas) ===")
    print(f"{'Ep':>3} {'Batch':>10} {'LossD':>8} {'LossG':>8} {'D(x)':>7} {'D(G(z))':>14}")
    t0 = time.time(); iters = 0

    for ep in range(epochs):
        for i, (real, _) in enumerate(loader):
            real = real.to(DEVICE); b = real.size(0)

            # --- Discriminador ---
            netD.zero_grad()
            lab = torch.full((b,), REAL_LABEL, device=DEVICE)
            out_real = netD(real)
            errD_real = criterion(out_real, lab); errD_real.backward()
            Dx = out_real.mean().item()

            noise = torch.randn(b, NZ, 1, 1, device=DEVICE)
            fake = netG(noise)
            lab.fill_(FAKE_LABEL)
            out_fake = netD(fake.detach())
            errD_fake = criterion(out_fake, lab); errD_fake.backward()
            DGz1 = out_fake.mean().item()
            errD = errD_real + errD_fake
            optD.step()

            # --- Gerador ---
            netG.zero_grad()
            lab.fill_(REAL_LABEL)
            out_fake2 = netD(fake)
            errG = criterion(out_fake2, lab); errG.backward()
            DGz2 = out_fake2.mean().item()
            optG.step()

            G_losses.append(errG.item()); D_losses.append(errD.item())
            if i % 50 == 0:
                print(f"{ep+1:>3} {i:>4}/{len(loader):<5} {errD.item():>8.3f} {errG.item():>8.3f} "
                      f"{Dx:>7.3f} {DGz1:>6.3f}/{DGz2:<6.3f}")
            if iters % 300 == 0 or (ep == epochs-1 and i == len(loader)-1):
                with torch.no_grad():
                    ff = netG(fixed_noise).detach().cpu()
                img_list.append(vutils.make_grid(ff, padding=2, normalize=True))
            iters += 1

    print(f"{tag} concluído em {(time.time()-t0)/60:.1f} min")
    return {"G_losses": G_losses, "D_losses": D_losses, "img_list": img_list, "fixed_noise": fixed_noise}

## 1. DCGAN-A — Baseline no Dataset Cru

Treinamos o primeiro DCGAN em todo o subconjunto cru. Ele serve de baseline **e** seu discriminador será o extrator de features da filtragem.

In [ ]:
hist_A = treinar_dcgan(netG_A, netD_A, dataloader_cru, EPOCHS_A, tag="DCGAN-A (cru)")

with torch.no_grad():
    fake_A = netG_A(torch.randn(64, NZ, 1, 1, device=DEVICE)).cpu()
plt.figure(figsize=(8,8)); plt.axis("off")
plt.title("DCGAN-A — amostras do dataset cru")
plt.imshow(np.transpose(vutils.make_grid(fake_A, padding=2, normalize=True), (1,2,0)))
plt.show()

## 2. Filtragem — "usar a IA para separar as imagens"

GAN não tem encoder, então usamos o **discriminador do DCGAN-A** como extrator: cada imagem real passa por `netD_A.extract()` e vira um vetor de 8192 features. Reduzimos com PCA e rodamos K-means. O cluster **mais denso** é o grupo concêntrico — instâncias com mesma proporção, forma e matiz.

In [ ]:
@torch.no_grad()
def extrair_features(netD, dataset, batch=256):
    netD.eval()
    ld = torch.utils.data.DataLoader(dataset, batch_size=batch, shuffle=False, num_workers=WORKERS)
    feats = []
    for x, _ in ld:
        feats.append(netD.extract(x.to(DEVICE)).cpu().numpy())
    netD.train()
    return np.concatenate(feats, axis=0)

print("Extraindo features com o discriminador do DCGAN-A...")
feats = extrair_features(netD_A, dataset_cru)
print(f"Features: {feats.shape}")

pca = PCA(n_components=50, random_state=SEED)
feats_red = pca.fit_transform(feats)
print(f"Variância explicada (50 comp.): {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
K = 8
km = KMeans(n_clusters=K, random_state=SEED, n_init=10).fit(feats_red)
labels = km.labels_; centros = km.cluster_centers_

densidades = []
for k in range(K):
    pts = feats_red[labels == k]
    if len(pts) == 0:
        densidades.append((k, 0, np.inf, 0.0)); continue
    dm = np.linalg.norm(pts - centros[k], axis=1).mean()
    densidades.append((k, len(pts), dm, 1.0/(dm+1e-8)))

print(f"{'Cluster':<9}{'Tamanho':<10}{'Dist.média':<13}{'Densidade':<10}")
print("-"*42)
for k, tam, dm, dens in sorted(densidades, key=lambda r: -r[3]):
    print(f"{k:<9}{tam:<10}{dm:<13.3f}{dens:<10.4f}")

In [ ]:
# Visualizar amostras de cada cluster
fig, axes = plt.subplots(K, 1, figsize=(14, 2.2*K))
for k in range(K):
    idx_k = np.where(labels == k)[0]
    sel = np.random.choice(idx_k, min(10, len(idx_k)), replace=False)
    imgs = torch.stack([dataset_cru[i][0] for i in sel])
    axes[k].imshow(np.transpose(vutils.make_grid(imgs, nrow=10, padding=2, normalize=True), (1,2,0)))
    dens_k = next(d for c,_,_,d in densidades if c==k)
    axes[k].set_title(f"Cluster {k} — {len(idx_k)} imagens — densidade {dens_k:.4f}", fontsize=10)
    axes[k].axis("off")
plt.tight_layout(); plt.savefig("clusters.png", dpi=110, bbox_inches="tight"); plt.show()

Escolhemos o cluster mais **denso** com tamanho mínimo razoável (GAN precisa de dados). Esse é o subconjunto concêntrico.

In [ ]:
MIN_TAM = 800
cands = sorted([(k,tam,dm,dens) for (k,tam,dm,dens) in densidades if tam >= MIN_TAM],
               key=lambda r: -r[3])
cluster_esc, tam_esc, _, dens_esc = cands[0]
print(f"Cluster concêntrico escolhido: #{cluster_esc} ({tam_esc} imagens, densidade {dens_esc:.4f})")

idx_conc = np.where(labels == cluster_esc)[0]
sel = np.random.choice(idx_conc, min(64, len(idx_conc)), replace=False)
imgs_conc = torch.stack([dataset_cru[i][0] for i in sel])
plt.figure(figsize=(8,8)); plt.axis("off")
plt.title(f"Cluster {cluster_esc} — grupo concêntrico")
plt.imshow(np.transpose(vutils.make_grid(imgs_conc, padding=2, normalize=True), (1,2,0)))
plt.show()

## 3. DCGAN-B — Treinamento no Subset Concêntrico

DCGAN novo, do zero, treinado **só no cluster filtrado**. Mesmo com menos dados, a baixa variabilidade deve gerar rostos mais nítidos e coerentes — *"falta de variabilidade nos formatos das figuras facilita"*.

In [ ]:
subset_conc = Subset(dataset_cru, idx_conc.tolist())
loader_conc = torch.utils.data.DataLoader(
    subset_conc, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=WORKERS, pin_memory=(DEVICE.type=="cuda"), drop_last=True,
)
print(f"Subset concêntrico: {len(subset_conc)} imagens | {len(loader_conc)} batches")

netG_B, netD_B = novo_par()
hist_B = treinar_dcgan(netG_B, netD_B, loader_conc, EPOCHS_B, tag="DCGAN-B (concêntrico)")

with torch.no_grad():
    fake_B = netG_B(torch.randn(64, NZ, 1, 1, device=DEVICE)).cpu()
plt.figure(figsize=(8,8)); plt.axis("off")
plt.title(f"DCGAN-B — amostras do cluster {cluster_esc}")
plt.imshow(np.transpose(vutils.make_grid(fake_B, padding=2, normalize=True), (1,2,0)))
plt.show()

## 4. Curvas de Loss

Oscilação é normal em GANs. O interessante é comparar a estabilidade: o DCGAN-B, com dados homogêneos, tende a curvas mais comportadas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for ax, hist, tit in [(axes[0], hist_A, "DCGAN-A (cru)"), (axes[1], hist_B, "DCGAN-B (concêntrico)")]:
    ax.plot(hist["G_losses"], label="G", color="#e74c3c", alpha=0.7, linewidth=0.8)
    ax.plot(hist["D_losses"], label="D", color="#3498db", alpha=0.7, linewidth=0.8)
    ax.set_title(f"Loss — {tit}"); ax.set_xlabel("Iterações"); ax.set_ylabel("BCELoss")
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig("loss_curves.png", dpi=100, bbox_inches="tight"); plt.show()

## 5. Comparação Final — DCGAN-A × DCGAN-B

In [ ]:
with torch.no_grad():
    fa = netG_A(torch.randn(64, NZ, 1, 1, device=DEVICE)).cpu()
    fb = netG_B(torch.randn(64, NZ, 1, 1, device=DEVICE)).cpu()

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].set_title("DCGAN-A (dataset cru)", fontsize=13); axes[0].axis("off")
axes[0].imshow(np.transpose(vutils.make_grid(fa, padding=2, normalize=True), (1,2,0)))
axes[1].set_title(f"DCGAN-B (concêntrico — cluster {cluster_esc})", fontsize=13); axes[1].axis("off")
axes[1].imshow(np.transpose(vutils.make_grid(fb, padding=2, normalize=True), (1,2,0)))
plt.tight_layout(); plt.savefig("comparativo_final.png", dpi=120, bbox_inches="tight"); plt.show()

## 6. Evolução Visual do Treinamento (DCGAN-B)

Usando o ruído fixo — os mesmos vetores z em todas as épocas — vemos o gerador aprender.

In [ ]:
img_list = hist_B["img_list"]
n_snap = min(len(img_list), 8)
idxs = np.linspace(0, len(img_list)-1, n_snap, dtype=int)

fig, axes = plt.subplots(2, 4, figsize=(16, 8)); axes = axes.flatten()
for j, ax in enumerate(axes):
    if j < len(idxs):
        ax.imshow(np.transpose(img_list[idxs[j]], (1,2,0)))
        ax.set_title(f"snapshot {idxs[j]+1}/{len(img_list)}", fontsize=9)
    ax.axis("off")
plt.suptitle("Evolução do Gerador B ao longo do treino", fontsize=15, y=1.01)
plt.tight_layout(); plt.savefig("evolucao_B.png", dpi=100, bbox_inches="tight"); plt.show()

## 7. Interpolação no Espaço Latente (DCGAN-B)

$$z_{interp} = (1-\alpha)\,z_1 + \alpha\,z_2,\quad \alpha\in[0,1]$$

Transições suaves indicam um espaço latente bem estruturado — favorecido pela homogeneidade do subset.

In [ ]:
netG_B.eval()
N_INTERP, N_PAIRS = 10, 4
fig, axes = plt.subplots(N_PAIRS, N_INTERP, figsize=(N_INTERP*1.6, N_PAIRS*1.6))
with torch.no_grad():
    for r in range(N_PAIRS):
        z1 = torch.randn(1, NZ, 1, 1, device=DEVICE)
        z2 = torch.randn(1, NZ, 1, 1, device=DEVICE)
        for c, a in enumerate(np.linspace(0, 1, N_INTERP)):
            z = (1-a)*z1 + a*z2
            img = netG_B(z).squeeze(0).cpu()
            img = (img*0.5+0.5).clamp(0,1)
            axes[r,c].imshow(np.transpose(img.numpy(), (1,2,0)))
            if r == 0: axes[r,c].set_title(f"α={a:.1f}", fontsize=8)
            axes[r,c].axis("off")
plt.suptitle("Interpolação no espaço latente — DCGAN-B", fontsize=14, y=1.02)
plt.tight_layout(); plt.savefig("interpolacao_B.png", dpi=100, bbox_inches="tight"); plt.show()
netG_B.train()

## 8. Salvando os Modelos

In [ ]:
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
torch.save({
    "G_A": netG_A.state_dict(), "D_A": netD_A.state_dict(),
    "G_B": netG_B.state_dict(), "D_B": netD_B.state_dict(),
    "cluster_escolhido": int(cluster_esc),
    "hparams": {"NZ":NZ,"NGF":NGF,"NDF":NDF,"NC":NC,"IMAGE_SIZE":IMAGE_SIZE,
                "LR":LR,"BETA1":BETA1,"BATCH_SIZE":BATCH_SIZE,
                "EPOCHS_A":EPOCHS_A,"EPOCHS_B":EPOCHS_B},
}, "/kaggle/working/checkpoints/dcgan_concentrico.pth")
print("Salvo em /kaggle/working/checkpoints/dcgan_concentrico.pth")

## Conclusões — voltando às anotações

| Anotação do professor | Como o projeto valida |
|---|---|
| *"Não adianta espremer na IA"* | DCGAN-A treinou em mais dados e produz amostras mais ruidosas que o DCGAN-B. |
| *"Identificar problema do dataset"* | O K-means sobre as features do discriminador revela os subgrupos naturais do dataset. |
| *"Não tem número mínimo, depende do problema"* | O subset concêntrico, bem menor, gera melhor no seu tipo-alvo. |
| *"Usar a IA para separar as imagens"* | O **discriminador do DCGAN-A** vira extrator de features para a clusterização — sem features feitas à mão. |
| *"Falta de variabilidade facilita"* | Curvas de loss mais estáveis e interpolação mais suave no DCGAN-B. |
| *"Mesma proporção, forma e matiz"* | É o que um cluster denso no espaço de features significa. |

### Diferença para a versão VAE
O VAE filtrava com o **encoder** (que ele tem nativamente). O DCGAN não tem encoder, então reaproveitamos o **discriminador** como extrator de representação — uma técnica clássica (o paper original do DCGAN mostra que as features do D são ótimas para classificação).

### Próximos passos
- Variar `K` e `SAMPLE_SIZE` para mapear o trade-off dados × qualidade.
- Aumentar `EPOCHS_B` (DCGANs melhoram bastante com 50–100 épocas).
- Re-filtrar usando o discriminador do DCGAN-B (curadoria recursiva).